In [3]:
import sys
import sklearn
import numpy as np
import os
import pandas as pd

In [2]:
!pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 1.0/100.2 MB 16.7 MB/s eta 0:00:06
   - -------------------------------------- 3.4/100.2 MB 9.6 MB/s eta 0:00:11
   -- ------------------------------------- 6.0/100.2 MB 10.5 MB/s eta 0:00:09
   --- ------------------------------------ 8.4/100.2 MB 10.8 MB/s eta 0:00:09
   ---- ----------------------------------- 10.5/100.2 MB 11.3 MB/s eta 0:00:08
   ----- ---------------------------------- 13.6/100.2 MB 11.6 MB/s eta 0:00:08
   ------ --------------------------------- 16.8/100.2 MB 12.3 MB/s eta 0:00:07
   -------- ------------------------------- 21.0/100.2 MB 13.4 MB/s eta 0:00:06
   --------- ------------------------------ 23.1/100.2 MB 12.8 MB/s eta 0:00:07
   ---------- ----------------------------- 25.2/100.2 MB 12.7 MB/s eta 0:00:06
   ----------- ---------------------------- 28.0/100.2 MB 12.5 MB/s eta 0:00:06
   ------------ --------------------------- 30.7/100.2


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
df = pd.read_csv('test.csv')
df.head()

,id,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,824,833,Lenovo,ThinkPad X1,Ultrabook,14.0,IPS Panel Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB Flash Storage,Intel HD Graphics 620,Windows 10,1.13kg
1,101,104,HP,15-bw000nv (E2-9000e/4GB/500GB/Radeon,Notebook,15.6,Full HD 1920x1080,AMD E-Series E2-9000e 1.5GHz,4GB,500GB HDD,AMD Radeon R2,Windows 10,2.1kg
2,1157,1175,Lenovo,ThinkPad T460s,Ultrabook,14.0,Full HD 1920x1080,Intel Core i7 6600U 2.6GHz,12GB,512GB SSD,Intel HD Graphics 520,Windows 7,1.4kg
3,940,954,HP,Omen 17-W006na,Gaming,17.3,IPS Panel Full HD 1920x1080,Intel Core i5 6300HQ 2.3GHz,8GB,128GB SSD + 1TB HDD,Nvidia GeForce GTX 965M,Windows 10,2.75kg
4,853,864,Lenovo,ThinkPad T470p,Ultrabook,14.0,IPS Panel Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia GeForce GT 940MX,Windows 10,1.96kg


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                391 non-null    int64  
 1   laptop_ID         391 non-null    int64  
 2   Company           391 non-null    object 
 3   Product           391 non-null    object 
 4   TypeName          391 non-null    object 
 5   Inches            391 non-null    float64
 6   ScreenResolution  391 non-null    object 
 7   Cpu               391 non-null    object 
 8   Ram               391 non-null    object 
 9   Memory            391 non-null    object 
 10  Gpu               391 non-null    object 
 11  OpSys             391 non-null    object 
 12  Weight            391 non-null    object 
dtypes: float64(1), int64(2), object(10)
memory usage: 39.8+ KB


In [1]:
import numpy as np      # Operaciones matemáticas (aquí se usa para la raíz cuadrada al calcular PPI)
import pandas as pd     
import re               # regres

# ==============================================================================
# FUNCIONES AUXILIARES 
# ==============================================================================

def parse_memory(mem_str):
    """
    Descompone descripciones como '128GB SSD + 1TB HDD' en columnas numéricas
    separadas por tipo de almacenamiento, todo normalizado a GB.
    """
    # Convierte a texto y limpia decimales innecesarios como '1.0TB' -> '1TB'
    mem_str = str(mem_str).replace('.0', '')
    
    # Diccionario inicializado en 0 para acumular la capacidad de cada tecnología
    storage = {'ssd_gb': 0, 'hdd_gb': 0, 'flash_gb': 0, 'hybrid_gb': 0}
    
    # Si el portátil tiene dos discos ('128GB SSD + 1TB HDD'), los separa en una lista
    parts = mem_str.split('+')
    
    for part in parts:
        part = part.strip()  # Elimina espacios sobrantes al inicio y al final
        
        # Busca un número (entero o decimal) seguido opcionalmente de espacio y 'GB' o 'TB'
        # Grupo 1: cantidad (ej. '128' o '1.5') | Grupo 2: unidad ('GB' o 'TB')
        match = re.search(r'(\d+(?:\.\d+)?)\s*(GB|TB)', part)
        if not match:
            continue  # Si ese fragmento no contiene un tamaño válido, salta al siguiente
            
        size = float(match.group(1))
        if match.group(2) == 'TB':
            size *= 1024  # Convierte terabytes a gigabytes para usar una única escala
        
        # Suma el tamaño a la tecnología correspondiente (usar '+=' soporta casos como '256GB SSD + 256GB SSD')
        part_upper = part.upper()
        if 'SSD' in part_upper:
            storage['ssd_gb'] += size
        elif 'HDD' in part_upper:
            storage['hdd_gb'] += size
        elif 'FLASH' in part_upper:
            storage['flash_gb'] += size
        elif 'HYBRID' in part_upper:
            storage['hybrid_gb'] += size
            
    # Bandera binaria: vale 1 si el portátil trae más de una unidad de almacenamiento, 0 si no
    storage['is_dual_storage'] = int(len(parts) > 1)
    
    # Devolver una Series hace que pandas expanda automáticamente las 5 claves en 5 columnas
    return pd.Series(storage)


def extract_cpu_tier(cpu):
    """
    Agrupa cientos de modelos exactos de CPU en gamas claras para evitar overfitting.
    El orden de los 'if' importa: se evalúan primero los modelos específicos antes que
    los genéricos ('intel' o 'amd').
    """
    cpu_l = str(cpu).lower()
    if 'xeon' in cpu_l: return 'Intel Xeon'       # Procesadores de estación de trabajo (muy caros)
    if 'i7' in cpu_l: return 'Intel i7'
    if 'i5' in cpu_l: return 'Intel i5'
    if 'i3' in cpu_l: return 'Intel i3'
    if 'core m' in cpu_l: return 'Intel Core M'   # Ultrabooks ligeros de bajo consumo
    if 'ryzen' in cpu_l: return 'AMD Ryzen'
    if 'intel' in cpu_l: return 'Intel Low'       # Captura Celeron, Pentium, Atom
    if 'amd' in cpu_l: return 'AMD Other'         # Captura series A, E, FX antiguas
    return 'Other'


def extract_gpu_tier(gpu):
    """
    Clasifica la tarjeta gráfica según su segmento de precio/rendimiento.
    Al igual que en CPU, el orden de prioridad evita solapamientos.
    """
    gpu_l = str(gpu).lower()
    # Gráficas profesionales certificadas (disparan drásticamente el precio)
    if 'quadro' in gpu_l or 'firepro' in gpu_l or 'radeon pro' in gpu_l:
        return 'Workstation'
    # Gráficas gaming dedicadas de Nvidia
    if 'gtx' in gpu_l or 'rtx' in gpu_l:
        return 'Gaming_Nvidia'
    # Gráficas gaming dedicadas de AMD
    if 'rx ' in gpu_l or 'radeon r9' in gpu_l:
        return 'Gaming_AMD'
    # Resto de gráficas dedicadas de gama entrada/media (ej. Nvidia MX150, 940MX, AMD R5/R7)
    if 'nvidia' in gpu_l or 'amd' in gpu_l:
        return 'Mid_Dedicated'
    # Todo lo demás (Intel HD/UHD/Iris Graphics, ARM Mali) son gráficas integradas
    return 'Integrated'


# ==============================================================================
# PIPELINE PRINCIPAL DE FEATURE ENGINEERING
# ==============================================================================

def feature_engineering(df):
    # Crea una copia para no modificar ni corromper el DataFrame original por fuera
    df = df.copy()
    
    # 1. Eliminar IDs
    # Borra columnas identificadoras solo si existen (evita errores si ya se eliminaron antes)
    df.drop(columns=[c for c in ['id', 'laptop_ID'] if c in df.columns], inplace=True)
    
    # 2. Conversión directa de texto a número
    # Quita el sufijo literal ('GB' y 'kg') y convierte a entero/flotante
    df['ram_gb'] = df['Ram'].str.replace('GB', '', regex=False).astype(int)
    df['weight_kg'] = df['Weight'].str.replace('kg', '', regex=False).astype(float)
    
    # 3. Descomposición de ScreenResolution
    # Extrae ancho y alto en píxeles (ej. de 'IPS Panel 1920x1080' saca 1920 y 1080)
    res = df['ScreenResolution'].str.extract(r'(\d+)x(\d+)').astype(float)
    df['res_x'], df['res_y'] = res[0], res[1]
    
    # Calcula PPI (píxeles por pulgada) usando Pitágoras: diagonal en píxeles / pulgadas físicas
    df['ppi'] = np.sqrt(df['res_x']**2 + df['res_y']**2) / df['Inches']
    
    # Crea variables binarias (1 = sí tiene la característica, 0 = no) ignorando mayúsculas/minúsculas
    df['is_touchscreen'] = df['ScreenResolution'].str.contains('Touchscreen', case=False).astype(int)
    df['is_ips'] = df['ScreenResolution'].str.contains('IPS', case=False).astype(int)
    df['is_4k'] = df['ScreenResolution'].str.contains('4K|3840x2160', case=False).astype(int)
    
    # 4. Descomposición de CPU
    # Extrae la velocidad del reloj justo antes de 'GHz' (ej. '2.8GHz' -> 2.8)
    df['cpu_ghz'] = df['Cpu'].str.extract(r'(\d+(?:\.\d+)?)GHz').astype(float)
    # Toma la primera palabra del texto como fabricante ('Intel', 'AMD', 'Samsung')
    df['cpu_brand'] = df['Cpu'].str.split().str[0]
    # Aplica la clasificación por gamas definida arriba
    df['cpu_tier'] = df['Cpu'].apply(extract_cpu_tier)
    # Detecta procesadores de alto voltaje/rendimiento (ej. '7700HQ', '6820HK', '8750H') frente a los 'U' de bajo consumo
    df['cpu_high_perf'] = df['Cpu'].str.contains(r'\b\d+[A-Z]*H[QK]?\b', regex=True).astype(int)
    
    # 5. Descomposición de Memory
    # Aplica parse_memory a cada fila; devuelve un DataFrame de 5 columnas nuevas
    mem_features = df['Memory'].apply(parse_memory)
    # Pega esas 5 columnas nuevas a la derecha del DataFrame principal (axis=1)
    df = pd.concat([df, mem_features], axis=1)
    # Suma todos los discos para saber el almacenamiento total en GB
    df['total_storage_gb'] = df[['ssd_gb', 'hdd_gb', 'flash_gb', 'hybrid_gb']].sum(axis=1)
    # Bandera binaria: indica si el equipo incluye almacenamiento rápido SSD
    df['has_ssd'] = (df['ssd_gb'] > 0).astype(int)
    
    # 6. Descomposición de GPU
    # Primera palabra del texto ('Nvidia', 'Intel', 'AMD', 'ARM')
    df['gpu_brand'] = df['Gpu'].str.split().str[0]
    # Clasificación por segmento (Workstation, Gaming, Dedicada media, Integrada)
    df['gpu_tier'] = df['Gpu'].apply(extract_gpu_tier)
    # Bandera binaria: 1 si tiene gráfica dedicada, 0 si es integrada
    df['is_dedicated_gpu'] = (df['gpu_tier'] != 'Integrated').astype(int)
    
    # 7. Agrupación de OpSys (Sistema Operativo)
    # Reduce categorías fragmentadas ('Windows 10', 'Windows 10 S', 'Windows 7') en 5 bloques limpios
    def group_os(os_name):
        os_l = str(os_name).lower()
        if 'windows' in os_l: return 'Windows'
        if 'mac' in os_l: return 'Mac'
        if 'linux' in os_l: return 'Linux'
        if 'chrome' in os_l: return 'ChromeOS'
        return 'NoOS_Other'
    df['os_group'] = df['OpSys'].apply(group_os)
    
    # 8. Detección de gamas premium en Product
    # Busca con operador OR ('|') nombres comerciales asociados a chasis/calidad de gama alta
    # na=False evita que posibles valores nulos rompan la conversión a entero
    premium_kw = 'ThinkPad|XPS|Precision|MacBook Pro|ZenBook|Spectre|EliteBook|ZBook|Alienware|ROG|Predator|Surface'
    df['is_premium_line'] = df['Product'].str.contains(premium_kw, case=False, na=False).astype(int)
    
    # 9. Variables de interacción (combinan columnas ya limpias)
    # Peso relativo al tamaño de pantalla: valores bajos identifican ultrabooks finos y caros
    df['weight_per_inch'] = df['weight_kg'] / df['Inches']
    # Proporción de almacenamiento que es SSD (el '+ 1' en el denominador evita divisiones entre cero)
    df['ssd_ratio'] = df['ssd_gb'] / (df['total_storage_gb'] + 1)
    
    # 10. Limpieza final
    # Elimina las columnas originales de texto crudo porque su información ya fue extraída
    raw_cols = ['Ram', 'Weight', 'ScreenResolution', 'Cpu', 'Memory', 'Gpu', 'OpSys', 'Product']
    df.drop(columns=raw_cols, inplace=True)
    
    return df

In [3]:
from sklearn.model_selection import train_test_split   # Divide los datos en entrenamiento y validación
from sklearn.metrics import mean_absolute_error        # Función para calcular el MAE (la métrica de Kaggle)
from catboost import CatBoostRegressor                 # Modelo de Gradient Boosting para regresión (predecir números)

# ------------------------------------------------------------------------------
# 1. Cargar archivos originales
# ------------------------------------------------------------------------------
df_train_full = pd.read_csv('train.csv')      # Datos con precios conocidos (para entrenar y validar)
df_kaggle_test = pd.read_csv('test.csv')      # Datos sin precio (los que tienes que predecir para Kaggle)

# ------------------------------------------------------------------------------
# 2. Preparar X, y y el test de Kaggle
# ------------------------------------------------------------------------------
# Guardamos la columna 'id' de test.csv ANTES de limpiarla, para reconstruir el submission al final
test_ids = df_kaggle_test['id'].copy()

# Aplicamos tu función de feature engineering a train (quitando primero la columna objetivo 'Price_euros')
X_full = feature_engineering(df_train_full.drop(columns=['Price_euros']))

# Separamos la variable objetivo (los precios reales que el modelo debe aprender a predecir)
y_full = df_train_full['Price_euros']

# Aplicamos exactamente la misma transformación al test de Kaggle para que tenga las mismas columnas
X_kaggle_test = feature_engineering(df_kaggle_test)

# Detectamos automáticamente qué columnas siguen siendo texto o categoría (ej. 'Company', 'cpu_tier', 'gpu_tier')
# para avisarle a CatBoost cuáles debe tratar como variables categóricas
cat_cols = X_full.select_dtypes(include=['object', 'category']).columns.tolist()

# ------------------------------------------------------------------------------
# 3. Sacar train y validación (tu "test local") a partir de train.csv
# ------------------------------------------------------------------------------
# Divide X_full e y_full en 80% entrenamiento (X_train, y_train) y 20% validación (X_val, y_val)
# random_state=42 fija la aleatoriedad para que siempre salga la misma partición y puedas comparar pruebas
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)

# ------------------------------------------------------------------------------
# 4. Entrenar y evaluar localmente
# ------------------------------------------------------------------------------
model = CatBoostRegressor(
    iterations=1000,      # Número máximo de árboles secuenciales que puede construir
    learning_rate=0.05,   # Tamaño del paso de aprendizaje (más bajo = aprende más despacio pero más fino)
    depth=6,              # Profundidad de cada árbol (6 es el equilibrio estándar; más alto aumenta riesgo de overfitting)
    loss_function='MAE',  # Le dice al modelo que minimice directamente el Error Absoluto Medio
    random_seed=42,       # Semilla fija para que el entrenamiento sea reproducible
    verbose=False         # Silencia la salida por pantalla de cada iteración (pon verbose=100 si quieres ver el progreso)
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,       # Indica qué columnas son categóricas para que CatBoost las procese nativamente
    eval_set=(X_val, y_val),     # Datos de validación que vigila en cada árbol para detectar overfitting
    early_stopping_rounds=100    # Si el error en X_val no mejora durante 100 árboles seguidos, detiene el entrenamiento
)

# Generamos predicciones sobre el 20% que el modelo NO usó para entrenar y medimos el error real en euros
val_preds = model.predict(X_val)
print(f"MAE en validación local: {mean_absolute_error(y_val, val_preds):.2f} €")

# ------------------------------------------------------------------------------
# 5. Reentrenar con el 100% de train.csv usando la mejor iteración encontrada
# ------------------------------------------------------------------------------
# Recupera en qué número de árbol se logró el MAE más bajo en validación
# (Nota: conviene sumar +1 porque el conteo empieza en 0)
best_iters = model.get_best_iteration() + 1

# Creamos un modelo nuevo con esa cantidad exacta de árboles, ya sin early stopping porque no habrá conjunto de validación
final_model = CatBoostRegressor(
    iterations=best_iters, learning_rate=0.05, depth=6,
    loss_function='MAE', random_seed=42, verbose=False
)

# Entrenamos usando TODOS los portátiles de train.csv para aprovechar el 100% de la información disponible
final_model.fit(X_full, y_full, cat_features=cat_cols)

# ------------------------------------------------------------------------------
# 6. Predecir sobre test.csv y generar el submission
# ------------------------------------------------------------------------------
# Predecimos los precios de los 391 portátiles de test.csv
kaggle_preds = final_model.predict(X_kaggle_test)

# Construimos el DataFrame final con los nombres de columna exactos que pide sample_submission.csv
submission = pd.DataFrame({'id': test_ids, 'Price_euros': kaggle_preds})

# Exportamos a CSV sin el índice de pandas (0, 1, 2...) para que Kaggle acepte el formato
submission.to_csv('submission.csv', index=False)

MAE en validación local: 184.45 €
